# Revisión de predicciones - Mundial 2026

Notebook para revisar, dataset por dataset, cómo han ido las predicciones que generé para el Mundial 2026.

In [1]:
import pandas as pd

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 50)

DATA_DIR = "../data"
PROCESSED_DIR = "../data/processed"

## 1. Calendario y resultados (`results.csv`)

Partidos históricos + calendario del Mundial 2026. Los partidos ya jugados tienen marcador; los que aún no se jugaron (o para los que todavía no cargué el resultado real) aparecen con `NaN` en `home_score`/`away_score`.

In [2]:
results = pd.read_csv(f"{DATA_DIR}/results.csv", parse_dates=["date"])

wc2026 = results[(results["tournament"] == "FIFA World Cup") & (results["date"] >= "2026-01-01")]
jugados = wc2026.dropna(subset=["home_score", "away_score"])
pendientes = wc2026[wc2026["home_score"].isna()]

print(f"Partidos del Mundial 2026 en el dataset: {len(wc2026)}")
print(f"  Con marcador real cargado: {len(jugados)}")
print(f"  Pendientes / sin marcador: {len(pendientes)}")

wc2026.head(20)

Partidos del Mundial 2026 en el dataset: 72
  Con marcador real cargado: 0
  Pendientes / sin marcador: 72


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49281,2026-06-11,Mexico,South Africa,NaN,NaN,FIFA World Cup,Mexico City,Mexico,False
49282,2026-06-11,South Korea,Czech Republic,NaN,NaN,FIFA World Cup,Zapopan,Mexico,True
49283,2026-06-12,Canada,Bosnia and Herzegovina,NaN,NaN,FIFA World Cup,Toronto,Canada,False
49284,2026-06-12,United States,Paraguay,NaN,NaN,FIFA World Cup,Inglewood,United States,False
49285,2026-06-13,Qatar,Switzerland,NaN,NaN,FIFA World Cup,Santa Clara,United States,True
49286,2026-06-13,Brazil,Morocco,NaN,NaN,FIFA World Cup,East Rutherford,United States,True
49287,2026-06-13,Haiti,Scotland,NaN,NaN,FIFA World Cup,Foxborough,United States,True
49288,2026-06-13,Australia,Turkey,NaN,NaN,FIFA World Cup,Vancouver,Canada,True
49289,2026-06-14,Germany,Curaçao,NaN,NaN,FIFA World Cup,Houston,United States,True
49290,2026-06-14,Ivory Coast,Ecuador,NaN,NaN,FIFA World Cup,Philadelphia,United States,True


## 2. Predicciones de marcador por partido de grupos (`score_predictions.csv`)

Marcador más probable, lambdas de Poisson y probabilidades W/E/D generadas por el modelo para cada partido de la fase de grupos.

In [3]:
score_predictions = pd.read_csv(f"{DATA_DIR}/score_predictions.csv")

print(f"Partidos con predicción de marcador: {len(score_predictions)}")
score_predictions

Partidos con predicción de marcador: 72


,grupo,local,visitante,marcador,lambda_local,lambda_visitante,p_local,p_empate,p_visitante
0,A,Mexico,South Africa,1-0,1.77,0.82,0.363,0.283,0.353
1,A,Mexico,South Korea,1-1,1.37,1.06,0.284,0.276,0.440
2,A,Mexico,Czech Republic,1-0,1.84,0.79,0.369,0.283,0.347
3,A,South Africa,South Korea,1-1,1.01,1.44,0.200,0.245,0.555
4,A,South Africa,Czech Republic,1-1,1.36,1.07,0.269,0.272,0.458
5,A,South Korea,Czech Republic,1-0,1.76,0.83,0.351,0.283,0.366
6,B,Canada,Bosnia and Herzegovina,2-0,2.36,0.62,0.448,0.275,0.278
7,B,Canada,Qatar,2-0,2.02,0.72,0.404,0.281,0.315
8,B,Canada,Switzerland,1-1,1.35,1.08,0.272,0.273,0.455
9,B,Bosnia and Herzegovina,Qatar,1-1,1.12,1.30,0.229,0.259,0.511


## 3. Combinaciones de mejores terceros (`combinaciones_terceros_mundial_2026.csv`)

Tabla de mapeo FIFA: para cada combinación posible de grupos que clasifican terceros, indica contra qué primero de grupo le toca jugar a cada tercero en dieciseisavos.

In [4]:
combinaciones_terceros = pd.read_csv(f"{DATA_DIR}/combinaciones_terceros_mundial_2026.csv")

print(f"Combinaciones posibles: {len(combinaciones_terceros)}")
combinaciones_terceros.head(10)

Combinaciones posibles: 495


,Combinacion_Id,Grupos_Clasificados,Grupos_Eliminados,Rival_de_1A,Rival_de_1B,Rival_de_1D,Rival_de_1E,Rival_de_1G,Rival_de_1I,Rival_de_1K,Rival_de_1L
0,1,"A,B,C,D,E,F,G,H","I,J,K,L",3C,3E,3B,3F,3A,3G,3D,3H
1,2,"A,B,C,D,E,F,G,I","H,J,K,L",3C,3E,3B,3F,3A,3G,3D,3I
2,3,"A,B,C,D,E,F,G,J","H,I,K,L",3C,3E,3B,3F,3A,3G,3D,3J
3,4,"A,B,C,D,E,F,G,K","H,I,J,L",3C,3E,3B,3F,3A,3G,3D,3K
4,5,"A,B,C,D,E,F,G,L","H,I,J,K",3C,3F,3B,3D,3A,3G,3L,3E
5,6,"A,B,C,D,E,F,H,I","G,J,K,L",3C,3E,3B,3A,3H,3F,3D,3I
6,7,"A,B,C,D,E,F,H,J","G,I,K,L",3C,3E,3B,3A,3H,3F,3D,3J
7,8,"A,B,C,D,E,F,H,K","G,I,J,L",3C,3E,3B,3A,3H,3F,3D,3K
8,9,"A,B,C,D,E,F,H,L","G,I,J,K",3C,3E,3B,3D,3A,3F,3L,3H
9,10,"A,B,C,D,E,F,I,J","G,H,K,L",3C,3E,3B,3A,3I,3F,3D,3J


## 4. Probabilidad de supervivencia por ronda (`processed/survival_probabilities.csv`)

Para cada selección, probabilidad (Monte Carlo) de llegar a cada ronda: Dieciseisavos, Octavos, Cuartos, Semis, Final y de ser Campeón.

In [5]:
survival_probabilities = pd.read_csv(f"{PROCESSED_DIR}/survival_probabilities.csv")

survival_probabilities.sort_values("Campeón", ascending=False).reset_index(drop=True)

,Team,Dieciseisavos,Octavos,Cuartos,Semis,Final,Campeón
0,Spain,0.91786,0.61114,0.44584,0.28252,0.21400,0.13396
1,England,0.81360,0.50054,0.30542,0.19490,0.11546,0.07134
2,France,0.75638,0.50874,0.35960,0.20638,0.11566,0.06650
3,Argentina,0.77732,0.46218,0.29510,0.16722,0.11164,0.06444
4,Morocco,0.85356,0.51602,0.29486,0.18172,0.08798,0.04874
5,Portugal,0.68238,0.36190,0.21362,0.12868,0.07388,0.04188
6,Japan,0.77960,0.44310,0.24262,0.13974,0.06280,0.03196
7,Colombia,0.81434,0.37260,0.20128,0.11188,0.05826,0.03178
8,Germany,0.73698,0.38968,0.20214,0.10316,0.05040,0.02822
9,Iran,0.79596,0.41570,0.24334,0.11226,0.06188,0.02734


## 5. Probabilidad de campeón - Monte Carlo (`processed/mc_results.csv`)

Resultado final de las simulaciones Monte Carlo: probabilidad de que cada selección gane el Mundial.

In [6]:
mc_results = pd.read_csv(f"{PROCESSED_DIR}/mc_results.csv")

mc_results["probability_%"] = (mc_results["probability"] * 100).round(2)
mc_results

,team,probability,probability_%
0,Spain,0.13396,13.40
1,England,0.07134,7.13
2,France,0.06650,6.65
3,Argentina,0.06444,6.44
4,Morocco,0.04874,4.87
5,Portugal,0.04188,4.19
6,Japan,0.03196,3.20
7,Colombia,0.03178,3.18
8,Germany,0.02822,2.82
9,Iran,0.02734,2.73


## 6. Ranking final de selecciones (`processed/ranking_final_2026.csv`)

Ratings finales (Elo, ataque, defensa, forma reciente y valor de mercado) usados como input de la simulación.

In [7]:
ranking_final_2026 = pd.read_csv(f"{PROCESSED_DIR}/ranking_final_2026.csv")

ranking_final_2026.sort_values("elo", ascending=False).reset_index(drop=True)

,team,elo,attack,defense,form,market_value_M
0,Spain,1879.392948,2.485377,0.313362,0.610331,1220
1,Morocco,1807.898122,1.657232,0.619785,0.750104,498
2,Argentina,1807.234855,3.647580,0.337163,0.336605,782
3,France,1794.437103,1.861400,0.557502,0.593099,1520
4,England,1776.709633,1.727491,0.498023,1.000000,1360
5,Japan,1766.980427,1.682633,0.676250,0.485626,271
6,Portugal,1746.385441,1.410739,0.669131,0.383447,1010
7,Germany,1736.505169,1.438746,0.660256,0.419746,947
8,Senegal,1734.231425,1.143253,0.782272,0.536533,478
9,Netherlands,1731.682386,1.689067,0.512533,0.631645,804


## 7. Histórico de ratings pre-partido (`processed/df_hist.csv`)

Elo/ODM/forma calculados partido a partido (pre-match) sobre el histórico usado para entrenar el modelo. Se muestran los últimos partidos calculados.

In [8]:
df_hist = pd.read_csv(f"{PROCESSED_DIR}/df_hist.csv", parse_dates=["date"])

print(f"Partidos en el histórico: {len(df_hist):,}")
print(f"Rango de fechas: {df_hist['date'].min().date()} a {df_hist['date'].max().date()}")
df_hist.tail(10)

Partidos en el histórico: 19,550
Rango de fechas: 2006-01-02 a 2026-05-31


,date,home_team,away_team,home_score,away_score,home_elo,away_elo,home_o,home_d,away_o,away_d,home_form,away_form,tournament,neutral,home_o_norm,home_d_norm,away_o_norm,away_d_norm
19540,2026-05-30,South Korea,Trinidad and Tobago,5,0,1694.059323,1469.930622,153.186066,12.068425,44.691156,57.570765,0.607963,-0.091234,Friendly,True,6.664116,0.670399,2.088965,2.910868
19541,2026-05-31,Poland,Ukraine,0,2,1560.454585,1570.212344,72.708069,19.794557,70.809074,23.493781,0.328691,0.034374,Friendly,False,3.163049,1.099584,3.309776,1.187882
19542,2026-05-31,Singapore,Mongolia,4,0,1420.424843,1269.159847,13.718572,95.864947,5.452515,321.504178,-0.614392,-0.641381,Friendly,False,0.596805,5.325280,0.254863,16.255752
19543,2026-05-31,Germany,Finland,4,0,1734.739973,1434.822186,160.720796,9.761059,46.475797,46.336823,0.419746,-0.368022,Friendly,False,6.991903,0.542225,2.172383,2.342862
19544,2026-05-31,Switzerland,Jordan,4,1,1673.005035,1638.167263,108.180548,14.213930,88.191774,28.607999,0.251148,0.153509,Friendly,False,4.706223,0.789581,4.122282,1.446465
19545,2026-05-31,Cape Verde,Serbia,3,0,1571.867927,1601.061589,48.353451,34.646139,71.648977,22.753271,0.461843,0.173303,Friendly,True,2.103540,1.924587,3.349035,1.150441
19546,2026-05-31,Brazil,Panama,6,2,1710.120342,1658.539355,329.318684,5.373534,92.851562,27.156734,0.194632,0.614475,Friendly,False,14.326486,0.298499,4.340091,1.373087
19547,2026-05-31,Japan,Iceland,1,0,1767.275333,1448.910839,187.965170,9.997518,41.517356,38.794921,0.485626,-0.326231,Friendly,False,8.177126,0.555360,1.940615,1.961532
19548,2026-05-31,Czech Republic,Kosovo,2,1,1580.982122,1598.269503,64.518775,23.309920,42.485613,43.556358,0.193528,0.329038,Friendly,False,2.806787,1.294862,1.985873,2.202277
19549,2026-05-31,United States,Senegal,3,2,1655.041864,1747.372724,205.451116,11.780421,127.711593,11.564923,0.156556,0.536533,Friendly,False,8.937825,0.654400,5.969527,0.584741


## 8. Set de validación (`processed/val_combined.csv`)

Partidos de Mundiales pasados (WC2018 + WC2022) usados para validar y calibrar el modelo antes de aplicarlo a 2026.

In [9]:
val_combined = pd.read_csv(f"{PROCESSED_DIR}/val_combined.csv", parse_dates=["date"])

print(f"Partidos de validación: {len(val_combined)}")
val_combined.head(10)

Partidos de validación: 211


,date,home_team,away_team,home_score,away_score,home_elo,away_elo,home_o,home_d,away_o,away_d,home_form,away_form,tournament,neutral
0,2018-06-14,Russia,Saudi Arabia,5,0,1547.191785,1573.916206,43.360432,13.355641,47.316031,18.918325,-0.288333,0.193562,FIFA World Cup,False
1,2018-06-15,Egypt,Uruguay,0,1,1614.291524,1636.853244,79.010005,8.542494,97.000954,9.809282,0.171941,0.165358,FIFA World Cup,True
2,2018-06-15,Morocco,Iran,0,1,1670.577292,1714.731564,56.057426,11.921764,100.317831,6.760355,0.324825,0.527678,FIFA World Cup,True
3,2018-06-15,Portugal,Spain,3,3,1758.596200,1765.307827,96.343118,6.809546,150.047014,3.883137,0.827485,0.945996,FIFA World Cup,True
4,2018-06-16,Croatia,Nigeria,2,0,1662.682411,1597.074105,57.093968,11.098294,49.009391,10.165636,0.363777,0.169464,FIFA World Cup,True
5,2018-06-16,France,Australia,2,1,1750.670089,1656.745003,98.172587,7.283570,71.899579,10.687586,0.623888,0.270451,FIFA World Cup,True
6,2018-06-16,Argentina,Iceland,1,1,1725.000393,1605.198452,136.715466,5.693110,31.482267,24.824100,0.121743,0.579071,FIFA World Cup,True
7,2018-06-16,Peru,Denmark,0,1,1696.499839,1640.767742,67.242348,16.258545,49.658087,11.508370,0.427771,0.557141,FIFA World Cup,True
8,2018-06-17,Costa Rica,Serbia,0,1,1609.203365,1589.991890,62.935128,17.370012,39.449855,15.577000,0.107967,0.473571,FIFA World Cup,True
9,2018-06-17,Germany,Mexico,0,1,1796.188510,1659.376328,119.224735,5.417175,93.468249,9.511201,0.921959,0.142821,FIFA World Cup,True
